# 얼굴 식별 최종본 v7: SCRFD + ArcFace + 오픈셋

이 노트북은 실시간 얼굴과 DB 임베딩을 비교하고, 미등록자는 `UNKNOWN`으로 반환합니다. 미등록자 이미지는 `C:\datasets\lfw_funneled`에서만 읽으며 별도로 촬영하지 않습니다.

최종 배포 모델 형식은 **ONNX**입니다. 현재 동의받은 데이터만으로 신경망을 안전하게 미세 조정하기에는 데이터가 부족합니다. 따라서 SCRFD와 ArcFace 사전학습 가중치는 고정하고, 운영 데이터에 맞는 유사도와 마진 임계값을 보정합니다.

1~6번을 순서대로 실행하세요. 7번 실시간 카메라 데모는 선택 사항입니다.

## 1. 환경과 CUDA 초기화

In [ ]:
from __future__ import annotations

import ctypes
import os
import time
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Any, Iterable

def find_project_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "deeplearning").is_dir() and (candidate / "webapps").is_dir():
            return candidate
    raise RuntimeError("smart_office_monitoring 저장소 안에서 실행하세요.")

def load_env_file(path: Path) -> None:
    if not path.is_file():
        return
    for raw in path.read_text(encoding="utf-8-sig").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = (item.strip() for item in line.split("=", 1))
        if len(value) >= 2 and value[0] == value[-1] and value[0] in {"'", '"'}:
            value = value[1:-1]
        os.environ.setdefault(key, value)

PROJECT_ROOT = find_project_root()
for path in (
    PROJECT_ROOT / "deeplearning/training/.env.face",
    PROJECT_ROOT / "webapps/fastapi/.env",
    PROJECT_ROOT / "webapps/fastapi/.env.local",
    PROJECT_ROOT / "deeplearning/training/.env",
    PROJECT_ROOT / "deeplearning/training/.env.local",
):
    load_env_file(path)

MODEL_ROOT = PROJECT_ROOT / "deeplearning/.models"
MONGODB_URI = os.environ.get("MONGODB_URI") or os.environ.get("DATABASE_URL", "")
MONGODB_DATABASE = (
    os.environ.get("MONGODB_DATABASE")
    or os.environ.get("DATABASE_NAME", "")
)
COLLECTION_NAME = os.environ.get("FACE_EMBEDDING_COLLECTION", "face_embeddings")
DETECTOR_PATH = Path(
    os.environ.get("FACE_DETECTION_MODEL_PATH")
    or MODEL_ROOT / "scrfd/scrfd_10g_bnkps.onnx"
).resolve()
RECOGNIZER_PATH = Path(
    os.environ.get("FACE_RECOGNITION_MODEL_PATH")
    or MODEL_ROOT / "buffalo_l/w600k_r50.onnx"
).resolve()
CAMERA_INDEX = int(os.environ.get("CAMERA_INDEX", "0"))
DETECTION_THRESHOLD = float(os.environ.get("FACE_DETECTION_THRESHOLD", "0.6"))
if not MONGODB_URI or not MONGODB_DATABASE:
    raise RuntimeError(
        "webapps/fastapi/.env 또는 training/.env.local에 "
        "DATABASE_URL과 DATABASE_NAME을 넣으세요."
    )
for path in (DETECTOR_PATH, RECOGNIZER_PATH):
    if not path.is_file():
        raise FileNotFoundError(path)

import cv2
import numpy as np
import torch

# Windows에서 ONNX Runtime이 PyTorch 번들 cuDNN을 확실히 찾게 한다.
TORCH_DLL_DIR = Path(torch.__file__).resolve().parent / "lib"
CUDNN_DLL = TORCH_DLL_DIR / "cudnn64_9.dll"
if os.name == "nt":
    if not CUDNN_DLL.is_file():
        raise FileNotFoundError(CUDNN_DLL)
    os.environ["PATH"] = f"{TORCH_DLL_DIR}{os.pathsep}{os.environ.get('PATH', '')}"
    _torch_dll_dir_handle = os.add_dll_directory(str(TORCH_DLL_DIR))
    _cudnn_handle = ctypes.WinDLL(str(CUDNN_DLL))

import onnxruntime as ort
from insightface.model_zoo import get_model
from insightface.utils import face_align
from pymongo import MongoClient
from pymongo.errors import PyMongoError

if not torch.cuda.is_available():
    raise RuntimeError("PyTorch CUDA를 사용할 수 없습니다.")
if "CUDAExecutionProvider" not in ort.get_available_providers():
    raise RuntimeError(f"CUDAExecutionProvider가 없습니다: {ort.get_available_providers()}")
print(
    f"Python {os.sys.version.split()[0]} | torch {torch.__version__} "
    f"| ORT {ort.__version__}"
)
print(f"CUDA {torch.version.cuda} | cuDNN {torch.backends.cudnn.version()}")
print(f"DB configured | collection={COLLECTION_NAME} | URI는 출력하지 않음")

## 2. DB 얼굴 갤러리 불러오기

In [ ]:
DIMENSION = 512
EXPECTED_METADATA = ("arcface", "insightface-buffalo_l-w600k_r50-v0.7",
                     "insightface-norm-crop-112-v1")


def normalize(value: Any) -> np.ndarray:
    vector = np.asarray(value, dtype=np.float32).reshape(-1)
    if vector.size != DIMENSION or not np.isfinite(vector).all():
        raise ValueError("유효한 512차원 embedding이 아닙니다.")
    norm = float(np.linalg.norm(vector))
    if norm <= 1e-12:
        raise ValueError("embedding norm이 0입니다.")
    return vector / norm


@dataclass(frozen=True)
class StudentEntry:
    student_id: str
    name: str
    number: str
    vector: np.ndarray


@dataclass(frozen=True)
class Gallery:
    entries: tuple[StudentEntry, ...]
    matrix: np.ndarray


def load_gallery() -> Gallery:
    client = MongoClient(
        MONGODB_URI, serverSelectionTimeoutMS=10_000, connectTimeoutMS=10_000)
    try:
        client.admin.command("ping")
        projection = {
            "_id": 0,
            "student_id": 1,
            "student_name": 1,
            "student_number": 1,
            "vector": 1,
            "dimension": 1,
            "normalized": 1,
            "model_name": 1,
            "model_version": 1,
            "preprocessing_version": 1,
        }
        entries = []
        seen = set()
        for doc in client[MONGODB_DATABASE][COLLECTION_NAME].find({}, projection):
            student_id = doc.get("student_id")
            if not isinstance(student_id, str) or not student_id or student_id in seen:
                raise RuntimeError("비어 있거나 중복된 student_id가 있습니다.")
            metadata = (doc.get("model_name"), doc.get(
                "model_version"), doc.get("preprocessing_version"))
            metadata_matches = (
                doc.get("dimension") == DIMENSION
                and doc.get("normalized") is True
                and metadata == EXPECTED_METADATA
            )
            if not metadata_matches:
                raise RuntimeError(f"{student_id}의 벡터 metadata가 현재 ArcFace와 다릅니다.")
            entries.append(StudentEntry(student_id, str(doc.get("student_name", "")), str(
                doc.get("student_number", "")), normalize(doc.get("vector"))))
            seen.add(student_id)
        if not entries:
            raise RuntimeError(f"{COLLECTION_NAME} 컬렉션이 비어 있습니다.")
        return Gallery(tuple(entries), np.stack([entry.vector for entry in entries]))
    except PyMongoError as exc:
        raise RuntimeError("MongoDB 연결/조회에 실패했습니다.") from exc
    finally:
        client.close()


gallery = load_gallery()
print(f"등록 학생 {len(gallery.entries)}명 로드 완료")
for entry in gallery.entries:
    print(f"- {entry.student_id} | {entry.number} | {entry.name}")


## 3. 얼굴 갤러리 품질 검사

In [ ]:
import json
from datetime import datetime, timezone

GALLERY_SUSPICIOUS_SIMILARITY = float(
    os.environ.get("GALLERY_SUSPICIOUS_SIMILARITY", "0.80"))
GALLERY_CRITICAL_SIMILARITY = float(
    os.environ.get("GALLERY_CRITICAL_SIMILARITY", "0.90"))
GALLERY_EXCLUDED_STUDENT_IDS = {item.strip() for item in os.environ.get(
    "GALLERY_EXCLUDED_STUDENT_IDS", "").split(",") if item.strip()}
GALLERY_REPORT_DIR = Path(os.environ.get("GALLERY_REPORT_DIR") or PROJECT_ROOT /
                          "deeplearning/training/runs/face_identification").resolve()
if not 0.0 <= GALLERY_SUSPICIOUS_SIMILARITY < GALLERY_CRITICAL_SIMILARITY <= 1.0:
    raise ValueError("gallery similarity 기준은 0 <= suspicious < critical <= 1이어야 합니다.")


def normalized_text(value: str) -> str:
    return "".join(value.lower().split())


def audit_gallery(current: Gallery) -> dict[str, Any]:
    pairs, similarities = [], []
    for left_index in range(len(current.entries)):
        for right_index in range(left_index + 1, len(current.entries)):
            left, right = current.entries[left_index], current.entries[right_index]
            similarity = float(current.matrix[left_index] @ current.matrix[right_index])
            similarities.append(similarity)
            same_number = bool(left.number and right.number and normalized_text(
                left.number) == normalized_text(right.number))
            same_name = bool(left.name and right.name and normalized_text(
                left.name) == normalized_text(right.name))
            if similarity < GALLERY_SUSPICIOUS_SIMILARITY and not same_number and not same_name:
                continue
            reasons = []
            if similarity >= GALLERY_SUSPICIOUS_SIMILARITY:
                reasons.append("high_similarity")
            if same_number:
                reasons.append("duplicate_student_number")
            if same_name:
                reasons.append("duplicate_student_name")
            is_critical = (
                similarity >= GALLERY_CRITICAL_SIMILARITY
                or same_number
                or same_name
            )
            severity = "critical" if is_critical else "suspicious"
            pairs.append(
                {
                    "left_student_id": left.student_id,
                    "right_student_id": right.student_id,
                    "cosine_similarity": similarity,
                    "severity": severity,
                    "reasons": reasons,
                }
            )
    norms = np.linalg.norm(current.matrix, axis=1)
    return {
        "gallery_count": len(current.entries),
        "dimension": int(current.matrix.shape[1]),
        "norm_min": float(norms.min()),
        "norm_max": float(norms.max()),
        "pairwise_similarity_min": (
            float(min(similarities)) if similarities else None
        ),
        "pairwise_similarity_max": (
            float(max(similarities)) if similarities else None
        ),
        "pairwise_similarity_mean": (
            float(np.mean(similarities)) if similarities else None
        ),
        "suspicious_threshold": GALLERY_SUSPICIOUS_SIMILARITY,
        "critical_threshold": GALLERY_CRITICAL_SIMILARITY,
        "flagged_pairs": sorted(
            pairs,
            key=lambda item: item["cosine_similarity"],
            reverse=True,
        ),
    }


def filter_gallery(current: Gallery, excluded_ids: set[str]) -> Gallery:
    known_ids = {entry.student_id for entry in current.entries}
    missing = sorted(excluded_ids - known_ids)
    if missing:
        raise ValueError(f"gallery에 없는 제외 student_id: {missing}")
    keep = [index for index, entry in enumerate(
        current.entries) if entry.student_id not in excluded_ids]
    if not keep:
        raise RuntimeError("모든 gallery 항목을 제외할 수 없습니다.")
    return Gallery(tuple(current.entries[index] for index in keep), current.matrix[keep].copy())


gallery_audit = audit_gallery(gallery)
GALLERY_REPORT_DIR.mkdir(parents=True, exist_ok=True)
gallery_report_path = GALLERY_REPORT_DIR / \
    f"gallery-audit-{datetime.now():%Y%m%d-%H%M%S}.json"
gallery_report = {"created_at": datetime.now(timezone.utc).isoformat(), **gallery_audit}
gallery_report_path.write_text(json.dumps(
    gallery_report, ensure_ascii=False, indent=2), encoding="utf-8")
critical_count = sum(item["severity"] ==
                     "critical" for item in gallery_audit["flagged_pairs"])
print(
    f"gallery={gallery_audit['gallery_count']}, "
    f"pairwise max={gallery_audit['pairwise_similarity_max']}"
)
print(f"flagged={len(gallery_audit['flagged_pairs'])}, critical={critical_count}")
for item in gallery_audit["flagged_pairs"]:
    print(
        f"[{item['severity']}] "
        f"{item['left_student_id']} <-> {item['right_student_id']} "
        f"sim={item['cosine_similarity']:.4f} "
        f"reasons={item['reasons']}"
    )
print(f"report: {gallery_report_path}")
gallery = filter_gallery(gallery, GALLERY_EXCLUDED_STUDENT_IDS)
print(
    f"active gallery: {len(gallery.entries)}명 "
    f"(로컬 제외 {len(GALLERY_EXCLUDED_STUDENT_IDS)}명, DB 변경 없음)"
)


## 4. ONNX Runtime CUDA로 SCRFD와 ArcFace 불러오기

In [ ]:
PROVIDERS = ["CUDAExecutionProvider", "CPUExecutionProvider"]
detector = get_model(str(DETECTOR_PATH), providers=PROVIDERS)
detector.prepare(ctx_id=0, input_size=(640, 640), det_thresh=DETECTION_THRESHOLD)
recognizer = get_model(str(RECOGNIZER_PATH), providers=PROVIDERS)
recognizer.prepare(ctx_id=0)
for label, model in (("SCRFD", detector), ("ArcFace", recognizer)):
    active = model.session.get_providers()
    if active[0] != "CUDAExecutionProvider":
        raise RuntimeError(f"{label}이 CUDA에서 실행되지 않습니다: {active}")
    print(label, active)
# provider 이름만 확인하지 않고 cuDNN Conv를 실제 실행한다.
detector.detect(np.zeros((640, 640, 3), dtype=np.uint8), max_num=0)
recognizer.get_feat(np.zeros((112, 112, 3), dtype=np.uint8))
print("SCRFD/ArcFace CUDA warm-up 성공")


## 5. 최종 추론 엔진

적용한 개선 사항: 얼굴 5점 정렬, 작거나 흐린 얼굴 거부, 원본과 좌우 반전 임베딩 평균, 유사도와 1·2위 마진 이중 판정, 시간축 다중 프레임 합의.

In [ ]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from deeplearning.face_identity import (
    FaceGallery,
    FaceIdentityEngine,
    GalleryEntry,
    IdentityThresholds,
    TemporalIdentityConsensus,
)

THRESHOLD_FILE = Path(os.environ["OPEN_SET_THRESHOLD_FILE"]).resolve()
threshold_data = json.loads(THRESHOLD_FILE.read_text(encoding="utf-8"))
thresholds = IdentityThresholds(
    similarity=float(threshold_data["similarity_threshold"]),
    margin=float(threshold_data["margin_threshold"]),
)
runtime_gallery = FaceGallery.from_entries(
    [
        GalleryEntry(entry.student_id, entry.vector)
        for entry in gallery.entries
    ]
)
engine = FaceIdentityEngine(
    detector=detector,
    recognizer=recognizer,
    gallery=runtime_gallery,
    thresholds=thresholds,
    detection_threshold=DETECTION_THRESHOLD,
    minimum_face_size=int(os.environ.get("FACE_MINIMUM_SIZE", "40")),
    minimum_blur_score=float(
        os.environ.get("FACE_MINIMUM_BLUR_SCORE", "20")
    ),
    use_flip_tta=(
        os.environ.get("FACE_USE_FLIP_TTA", "true").lower() == "true"
    ),
    tta_similarity_band=2.0,
    tta_margin_band=2.0,
)

print(
    f"임계값={thresholds.similarity:.4f}/{thresholds.margin:.4f}; "
    f"등록 학생={len(gallery.entries)}명"
)


## 6. LFW 미등록자 독립 평가와 최종 산출물

확정된 임계값을 사용하며, 정렬된 LFW 경로에서 처음 3,000개 파일을 제외한 구간을 평가합니다. v4 보정은 앞쪽에서 후보 1,148개를 확인해 유효 샘플 1,000개를 확보했으므로 최종 평가 구간과 분리됩니다. 실행하면 `metrics.json`, 예측 CSV, 모델 명세와 PNG 그래프 3개가 생성됩니다.

In [ ]:
import csv
from datetime import datetime, timezone

import matplotlib.pyplot as plt

LFW_ROOT = Path(
    os.environ.get(
        "OPEN_SET_UNKNOWN_DATASET_DIR",
        r"C:\datasets\lfw_funneled",
    )
).resolve()
FINAL_COUNT = int(os.environ.get("OPEN_SET_FINAL_UNKNOWN_SAMPLES", "1000"))
RUN_ROOT = Path(
    os.environ.get("OPEN_SET_OUTPUT_DIR")
    or PROJECT_ROOT / "deeplearning/training/runs/face_identification"
)
run_dir = RUN_ROOT / f"v7-final-{datetime.now():%Y%m%d-%H%M%S}"
run_dir.mkdir(parents=True, exist_ok=False)

paths = [
    path
    for path in LFW_ROOT.rglob("*")
    if path.suffix.lower() in {".jpg", ".jpeg", ".png"}
]
paths.sort()
paths = paths[3000:]

rows = []
unreadable = 0
rejected = 0
latency = []

for path in paths:
    image = cv2.imread(str(path))
    if image is None:
        unreadable += 1
        continue

    started = time.perf_counter()
    detections = engine.identify(image)
    latency.append((time.perf_counter() - started) * 1000)

    if len(detections) != 1:
        rejected += 1
        continue

    detection = detections[0]
    rows.append(
        {
            "file": str(path.relative_to(LFW_ROOT)),
            "accepted": detection.student_id is not None,
            "similarity": detection.similarity,
            "margin": detection.margin,
            "quality": detection.quality,
            "reason": detection.rejected_reason or "accepted",
        }
    )
    if len(rows) % 100 == 0:
        print(f"미등록자 평가 {len(rows)}/{FINAL_COUNT}", end="\r")
    if len(rows) >= FINAL_COUNT:
        break

if len(rows) < FINAL_COUNT:
    raise RuntimeError(
        f"유효한 미등록자 이미지가 부족합니다: {len(rows)}/{FINAL_COUNT}"
    )

false_accepts = sum(row["accepted"] for row in rows)
far = false_accepts / len(rows)
registered_files = sorted(RUN_ROOT.glob("registered-evaluation-*.json"))
registered = (
    json.loads(registered_files[-1].read_text(encoding="utf-8"))
    if registered_files
    else None
)
known_accuracy = (
    registered["metrics"]["top1_accuracy"] if registered else None
)

summary = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "model_format": "onnx",
    "models": {
        "detector": DETECTOR_PATH.name,
        "recognizer": RECOGNIZER_PATH.name,
    },
    "gallery_count": len(gallery.entries),
    "embedding_dimension": 512,
    "thresholds": {
        "similarity": thresholds.similarity,
        "margin": thresholds.margin,
    },
    "unknown": {
        "dataset": str(LFW_ROOT),
        "samples": len(rows),
        "false_accepts": false_accepts,
        "far": far,
        "unreadable": unreadable,
        "rejected_before_sample": rejected,
    },
    "known": {
        "source": str(registered_files[-1]) if registered else None,
        "top1_accuracy": known_accuracy,
        "note": "기존 촬영 평가 결과이며 새로 촬영하지 않음",
    },
    "latency_ms": {
        "mean": float(np.mean(latency)),
        "p95": float(np.percentile(latency, 95)),
    },
    "limitations": [
        "LFW와 실제 강의실 데이터 분포가 다름",
        "최신 등록자 평가는 학생 한 명 중심임",
        "시간축 합의 효과는 저장 영상으로 별도 평가해야 함",
    ],
}
(run_dir / "metrics.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

with (run_dir / "unknown_predictions.csv").open(
    "w",
    newline="",
    encoding="utf-8-sig",
) as output_file:
    writer = csv.DictWriter(output_file, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)

manifest = {
    "format": "onnx",
    "detector": str(DETECTOR_PATH),
    "recognizer": str(RECOGNIZER_PATH),
    "threshold_file": str(THRESHOLD_FILE),
    "preprocessing": (
        "SCRFD 5-point -> norm_crop 112 -> ArcFace -> L2 -> flip TTA"
    ),
    "output": (
        "student_id|null, bbox, detection_confidence, similarity, "
        "margin, quality, rejected_reason"
    ),
}
(run_dir / "model-manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

graph_settings = [
    ("similarity", thresholds.similarity, "unknown-similarity.png"),
    ("margin", thresholds.margin, "unknown-margin.png"),
]
for field, threshold, file_name in graph_settings:
    figure, axis = plt.subplots(figsize=(7, 4))
    axis.hist([row[field] for row in rows], bins=40)
    axis.axvline(threshold, color="red", label="판정 임계값")
    field_label = {
        "similarity": "유사도",
        "margin": "1·2위 마진",
    }[field]
    axis.set(
        title=f"LFW 미등록자 {field_label} 분포",
        xlabel=field_label,
        ylabel="이미지 수",
    )
    axis.legend()
    figure.tight_layout()
    figure.savefig(run_dir / file_name, dpi=160)
    plt.show()

figure, axis = plt.subplots(figsize=(6, 4))
values = [known_accuracy or 0, 1 - far]
bars = axis.bar(["등록자 Top-1 정확도", "미등록자 거부율"], values)
axis.set_ylim(0, 1.05)
axis.bar_label(bars, fmt="%.3f")
axis.set_ylabel("비율")
figure.tight_layout()
figure.savefig(run_dir / "final-rates.png", dpi=160)
plt.show()

print(json.dumps(summary, ensure_ascii=False, indent=2))
print(f"최종 산출물: {run_dir}")


## 7. 선택 사항: 실시간 카메라 데모

이 셀은 단일 카메라 확인용입니다. 운영 환경에서는 worker가 프레임 수집과 추적 ID를 관리하고 `engine.identify(frame)`을 호출한 다음, 문서화된 HTTP 계약으로 결과를 FastAPI에 전달합니다. 이 간단한 데모는 다중 인물 추적기를 대신하지 않습니다.

In [ ]:
def run_realtime_demo() -> None:
    camera = cv2.VideoCapture(
        CAMERA_INDEX,
        cv2.CAP_DSHOW if os.name == "nt" else cv2.CAP_ANY,
    )
    consensus = TemporalIdentityConsensus(
        window_size=5,
        consensus_count=4,
    )
    if not camera.isOpened():
        camera.release()
        raise RuntimeError("카메라를 열지 못했습니다.")

    try:
        while True:
            ok, frame = camera.read()
            if not ok:
                continue

            for index, detection in enumerate(engine.identify(frame)):
                stable_id = consensus.update(
                    str(index),
                    detection.student_id,
                )
                if stable_id is None:
                    label = "UNKNOWN"
                    color = (0, 0, 255)
                else:
                    label = next(
                        (
                            entry.name
                            for entry in gallery.entries
                            if entry.student_id == stable_id
                        ),
                        stable_id,
                    )
                    color = (0, 200, 0)

                left, top, right, bottom = detection.bbox
                cv2.rectangle(frame, (left, top), (right, bottom), color, 2)
                cv2.putText(
                    frame,
                    (
                        f"{label} s={detection.similarity:.3f} "
                        f"m={detection.margin:.3f}"
                    ),
                    (left, max(25, top - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.55,
                    color,
                    2,
                )

            cv2.imshow("얼굴 식별 v7", frame)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break
    finally:
        camera.release()
        cv2.destroyAllWindows()


run_realtime_demo()


## 8. 다른 모델로 교체하는 방법

- 얼굴 검출기 어댑터는 `detect(image, max_num=0)` 호출에서 얼굴 위치·신뢰도와 5점 랜드마크를 반환해야 합니다.
- 얼굴 인식기 어댑터는 `get_feat(112x112 BGR)` 호출에서 임베딩을 반환해야 합니다.
- 인식 모델, 전처리 또는 임베딩 차원이 바뀌면 DB의 모든 얼굴 임베딩을 다시 생성해야 합니다.
- 모델 버전이 달라지면 기존 임계값을 재사용하지 말고 등록자·미등록자 세트로 다시 보정하고 독립 평가해야 합니다.
- PT는 학습 체크포인트로 유용합니다. 현재 ONNX Runtime·Windows 환경과 향후 Jetson 적용 경로에서는 ONNX를 최종 배포 산출물로 선택합니다.